# Relative Value Difference Bias Check

This notebook will recover parameters from simulated data of three groups: large left preference, neutral perference, and large right preference. The underlying hypothesis is that parameter recovery bias (high noise, low drift) seems to be a result of a structural bias in the novel simulation methodology. Without quantifying this bias, which is believed to be conditional on relative value differences, the following work will only investigate the issues and determine if the bias is skewed towards large absolute value relative value differences if the bias is positionally dependent (left fixation is more likely).

## Stratifying Data

Define 
$$
\Delta_i = RDV_i = V_{L,i} - V_{R,i}.
$$

Then
$$
\mathcal{T}_R(\tau) = \{i:\Delta_i\le -\tau\},\\
\mathcal{T}_0(\epsilon) = \{i:|\Delta_i|\le\epsilon\},\\
\mathcal{T}_L(\tau) = \{i:\Delta_i\ge \tau\},
$$
and set $\tau = 3.5$ and $\epsilon = 0.5$.

In [ ]:
# If the notebook is moved into /exploratory_notebooks, then uncomment the code below

# import sys, os

# sys.path.insert(0, os.path.abspath(".."))

In [ ]:
import pandas as pd
from ast import literal_eval

df_raw = pd.read_csv('1ms_trial_data.csv')
df_raw['RT'] = df_raw['RT']*1000 # adjustment for RT
df_raw['fixation'] = df_raw['fixation'].apply(literal_eval)

to_drop = pd.read_csv("dropped_trials.csv").rename(columns={"parcode": "sub_id"})

df = df_raw.loc[
    ~df_raw.set_index(["sub_id", "trial"]).index.isin(
        to_drop.set_index(["sub_id", "trial"]).index
    )
    & (~df_raw["hidden"])
]


df.head()

In [ ]:
large_rvd_threshold = 3.5
neutral_rvd_threshold = 0.5

right_pref_df = df.loc[df["avgWTP_left"] - df["avgWTP_right"] <= -large_rvd_threshold]
neutral_pref_df = df.loc[abs(df["avgWTP_left"] - df["avgWTP_right"] <= 0.5)]
left_pref_df = df.loc[df["avgWTP_left"] - df["avgWTP_right"] >= large_rvd_threshold]

A quick reference to model-free analysis, the average response times of trials with large preferences to either stimuli should be shorter than trials with neutral perferences.

In [ ]:
import numpy as np

print(f'Trials with large right preference: {right_pref_df.shape[0]} (average RT: {np.mean(right_pref_df['RT']):.2f} ms)')
print(f'Trials with neutral preference: {neutral_pref_df.shape[0]} (average RT: {np.mean(neutral_pref_df['RT']):.2f} ms)')
print(f'Trials with large left preference: {left_pref_df.shape[0]} (average RT: {np.mean(left_pref_df['RT']):.2f} ms)')

## Simulating Data

In [ ]:
from simulation import get_corrected_empirical_distributions

legend = {
    "left": {1},
    "right": {2},
    "transition": {0}, 
    "blank_fixation": {4}
}
fixation_col = 'fixation'
left_value_col = 'avgWTP_left'
right_value_col = 'avgWTP_right'
cutoff = 0.95

right_value_diffs = np.arange(-4, -3.25, 0.25)
right_empirical_distributions = get_corrected_empirical_distributions(
    right_pref_df,
    value_diffs=right_value_diffs,
    legend=legend,
    fixation_col=fixation_col,
    left_value_col=left_value_col,
    right_value_col=right_value_col,
    cutoff=cutoff
)

neutral_value_diffs = np.arange(-0.5, 0.75, 0.25)
neutral_empirical_distributions = get_corrected_empirical_distributions(
    neutral_pref_df,
    value_diffs=neutral_value_diffs,
    legend=legend,
    fixation_col=fixation_col,
    left_value_col=left_value_col,
    right_value_col=right_value_col,
    cutoff=cutoff
)

left_value_diffs = np.arange(3.5, 4.25, 0.25)
left_empirical_distributions = get_corrected_empirical_distributions(
    left_pref_df,
    value_diffs=left_value_diffs,
    legend=legend,
    fixation_col=fixation_col,
    left_value_col=left_value_col,
    right_value_col=right_value_col,
    cutoff=cutoff
)

In [ ]:
right_trials = right_pref_df[['avgWTP_left', 'avgWTP_right']].sample(n=300, random_state=42)
right_trials['fixation'] = None

neutral_trials = neutral_pref_df[['avgWTP_left', 'avgWTP_right']].sample(n=300, random_state=42)
neutral_trials['fixation'] = None

left_trials = left_pref_df[['avgWTP_left', 'avgWTP_right']].sample(n=300, random_state=42)
left_trials['fixation'] = None

In [ ]:
from simulation import generate_fixations

dt = 0.01

right_trials_dict = []
for row in right_trials.itertuples(index=False):
    fx = generate_fixations(
        dt,
        row.avgWTP_left - row.avgWTP_right,
        right_empirical_distributions
    )
    if fx is not None:
        right_trials_dict.append({
            "avgWTP_left": row.avgWTP_left,
            "avgWTP_right": row.avgWTP_right,
            "fixation": fx
        })

neutral_trials_dict = []
for row in neutral_trials.itertuples(index=False):
    fx = generate_fixations(
        dt,
        row.avgWTP_left - row.avgWTP_right,
        neutral_empirical_distributions
    )
    if fx is not None:
        neutral_trials_dict.append({
            "avgWTP_left": row.avgWTP_left,
            "avgWTP_right": row.avgWTP_right,
            "fixation": fx
        })

left_trials_dict = []
for row in left_trials.itertuples(index=False):
    fx = generate_fixations(
        dt,
        row.avgWTP_left - row.avgWTP_right,
        left_empirical_distributions
    )
    if fx is not None:
        left_trials_dict.append({
            "avgWTP_left": row.avgWTP_left,
            "avgWTP_right": row.avgWTP_right,
            "fixation": fx
        })

In [ ]:
from simulation import simulate

seed = 42
model_conditions = {'drift_rate': 0.8, 'theta': 0.5, 'noise': 0.63}

right_results_df = simulate(dt, model_conditions, right_trials_dict, seed=seed, save_results=False)

right_results_df['sub_id'] = f'seed{seed}_preferenceR_sim'
right_results_df['trial'] = range(1, len(right_trials_dict) + 1)
right_results_df = right_results_df.rename(columns={'fixation': 'fix_sequence'})
right_results_df = right_results_df.drop(columns = ['trajectory'])
print(f'Average RT: {np.average(right_results_df.loc[:, "RT"]):.3f} seconds')
right_results_df.head()

In [ ]:
neutral_results_df = simulate(dt, model_conditions, neutral_trials_dict, seed=seed, save_results=False)

neutral_results_df['sub_id'] = f'seed{seed}_preferenceN_sim'
neutral_results_df['trial'] = range(1, len(neutral_trials_dict) + 1)
neutral_results_df = neutral_results_df.rename(columns={'fixation': 'fix_sequence'})
neutral_results_df = neutral_results_df.drop(columns = ['trajectory'])
print(f'Average RT: {np.average(neutral_results_df.loc[:, "RT"]):.3f} seconds')
neutral_results_df.head()

In [ ]:
left_results_df = simulate(dt, model_conditions, left_trials_dict, seed=seed, save_results=False)

left_results_df['sub_id'] = f'seed{seed}_preferenceL_sim'
left_results_df['trial'] = range(1, len(left_trials_dict) + 1)
left_results_df = left_results_df.rename(columns={'fixation': 'fix_sequence'})
left_results_df = left_results_df.drop(columns = ['trajectory'])
print(f'Average RT: {np.average(left_results_df.loc[:, "RT"]):.3f} seconds')
left_results_df.head()